# 05 — Cached Generation Evaluation

Official cached evaluation workflow for HaluGuard.

This notebook reuses existing artifacts only. It does not recompute embeddings and does not retrain models. Start with `HALUGUARD_LIMIT=10`, inspect the tables, then scale to 100-200 examples.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np
import torch

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(os.environ.get('HALUGUARD_ROOT', Path.cwd())).resolve()
if not (ROOT / 'haluguard').exists() and (Path.cwd().parent / 'haluguard').exists():
    ROOT = Path.cwd().parent.resolve()
DATA = Path(os.environ.get('HALUGUARD_DATA_DIR', ROOT / 'data')).resolve()
RESULTS = DATA / 'results' / 'cached_eval'
DETAILS = RESULTS / 'details'
CHECKPOINT_DIRS = []

for path in (RESULTS, DETAILS):
    path.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LIMIT = int(os.environ.get('HALUGUARD_LIMIT', '10'))
BATCH_SIZE = int(os.environ.get('HALUGUARD_BATCH_SIZE', '8'))
TOP_K = int(os.environ.get('HALUGUARD_TOP_K', '5'))
FAKE_GENERATOR = os.environ.get('HALUGUARD_FAKE_GENERATOR', '1') == '1'
FAKE_DATASET = os.environ.get('HALUGUARD_FAKE_DATASET', '1' if FAKE_GENERATOR else '0') == '1'

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('root       =', ROOT)
print('data       =', DATA)
print('results    =', RESULTS)
print('device     =', DEVICE)
print('limit      =', LIMIT)
print('fake gen   =', FAKE_GENERATOR)

## 2. Artifact Manifest

This cell must pass before any generation cells run. RepoBench cached artifacts are required; CrossCodeEval embeddings are optional.

In [ ]:
from haluguard.artifacts import (
    build_artifact_manifest,
    discover_checkpoint_dirs,
    load_cceval_cached_artifacts,
    load_repobench_cached_artifacts,
    validate_repobench_cached_artifacts,
)

CHECKPOINT_DIRS = discover_checkpoint_dirs(ROOT)
manifest = build_artifact_manifest(DATA, CHECKPOINT_DIRS)
try:
    import pandas as pd
    display(pd.DataFrame(manifest))
except ImportError:
    for row in manifest:
        print(row)

validate_repobench_cached_artifacts(DATA)
repobench_artifacts = load_repobench_cached_artifacts(DATA, map_location='cpu')
cceval_artifacts = load_cceval_cached_artifacts(DATA, map_location='cpu')
print('RepoBench cached artifacts loaded')
print('CrossCodeEval cached embeddings:', 'yes' if cceval_artifacts is not None else 'no')

## 3. Generator and Optional Semantic Metric Encoder

Fake-generator mode is for local notebook checks. Set `HALUGUARD_FAKE_GENERATOR=0` in Colab for real DeepSeek-Coder generation.

In [ ]:
from haluguard.eval_matrix import fake_generator

metric_encoder = None
metric_tokenizer = None

if FAKE_GENERATOR:
    generator = fake_generator
    print('Using fake generator for a fast local smoke check.')
else:
    from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer
    from haluguard.generate import generate_next_line_batch

    GEN_NAME = os.environ.get('HALUGUARD_GEN_NAME', 'deepseek-ai/deepseek-coder-1.3b-base')
    print('Loading generator:', GEN_NAME)
    gen_tokenizer = AutoTokenizer.from_pretrained(GEN_NAME, trust_remote_code=True)
    gen_model = AutoModelForCausalLM.from_pretrained(
        GEN_NAME,
        torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
        device_map='auto' if DEVICE == 'cuda' else None,
        trust_remote_code=True,
    ).eval()
    if gen_tokenizer.pad_token_id is None:
        gen_tokenizer.pad_token_id = gen_tokenizer.eos_token_id

    def generator(prompts):
        return generate_next_line_batch(
            prompts,
            tokenizer=gen_tokenizer,
            model=gen_model,
            device=DEVICE,
            max_new_tokens=64,
            temperature=0.2,
            max_prompt_tokens=2048,
        )

    if os.environ.get('HALUGUARD_CODEBERT_SCORE', '1') == '1':
        metric_tokenizer = AutoTokenizer.from_pretrained('microsoft/codebert-base')
        metric_encoder = AutoModel.from_pretrained('microsoft/codebert-base').to(DEVICE).eval()
        for param in metric_encoder.parameters():
            param.requires_grad_(False)
        print('CodeBERTScore encoder loaded')

## 3.5 CrossCodeEval Cache Status

This notebook is strictly cached-only. It never computes CrossCodeEval embeddings. If cached CrossCodeEval artifacts are missing or incompatible, the relevant cosine/HaluGuard rows stay skipped.

In [ ]:
cceval_meta = (cceval_artifacts or {}).get('meta', {}) if cceval_artifacts is not None else {}

if FAKE_DATASET:
    print('Fake dataset mode: CrossCodeEval cache remains optional.')
elif cceval_artifacts is None:
    print('No cached CrossCodeEval embeddings found — cosine/HaluGuard transfer rows will be skipped.')
else:
    print('Using cached CrossCodeEval embeddings only')
    print(json.dumps(cceval_meta, indent=2, sort_keys=True))


## 4. Method Availability

In [ ]:
from haluguard.eval_matrix import build_method_plan

repobench_plans = build_method_plan('repobench', DATA, CHECKPOINT_DIRS)
cceval_plans = build_method_plan('crosscodeeval', DATA, CHECKPOINT_DIRS)
method_rows = [p.to_row() for p in repobench_plans + cceval_plans]

try:
    import pandas as pd
    display(pd.DataFrame(method_rows))
except ImportError:
    for row in method_rows:
        print(row)

skips = [row for row in method_rows if row['status'] != 'ready']
(RESULTS / 'method_skips.json').write_text(json.dumps(skips, indent=2), encoding='utf-8')
print('skip rows:', len(skips))

## 5. Load Benchmarks

In [ ]:
from typing import Iterator, Optional

from haluguard.benchmarks.base import Example
from haluguard.benchmarks.repobench import RepoBenchLoader
from haluguard.benchmarks.crosscodeeval import CrossCodeEvalLoader

class _TinyLoader:
    def __init__(self, name):
        self.name = name
        self._examples = [
            Example(
                cropped_code='def f(x):\n    return',
                context_chunks=[
                    {'snippet': 'def helper(x):\n    return x + 1', 'path': 'helpers.py'},
                    {'snippet': 'class Client:\n    pass', 'path': 'client.py'},
                ],
                gold_index=0 if name == 'repobench' else None,
                reference='return',
                import_statement='',
                metadata={'source_index': 0, 'task_id': f'{name}-0'},
            )
        ]

    def iter_examples(self, limit: Optional[int] = None) -> Iterator[Example]:
        count = 0
        for example in self._examples:
            if limit is not None and count >= limit:
                return
            yield example
            count += 1

if FAKE_DATASET:
    repobench_loader = _TinyLoader('repobench')
    cceval_loader = _TinyLoader('crosscodeeval')
    print('using tiny in-memory benchmark loaders')
else:
    repobench_loader = RepoBenchLoader()
    cceval_loader = CrossCodeEvalLoader()
    print('benchmark loaders ready')

## 6. RepoBench Retrieval Table

In [ ]:
from haluguard.eval_matrix import compute_repobench_retrieval_table, write_table_files

repobench_retrieval_rows = compute_repobench_retrieval_table(
    plans=repobench_plans,
    loader=repobench_loader,
    repobench_artifacts=repobench_artifacts,
    checkpoint_dirs=CHECKPOINT_DIRS,
    limit=LIMIT,
    top_k=TOP_K,
    device=DEVICE,
)
write_table_files(repobench_retrieval_rows, RESULTS / 'repobench_retrieval_table')
try:
    import pandas as pd
    display(pd.DataFrame(repobench_retrieval_rows))
except ImportError:
    print(repobench_retrieval_rows)

## 7. Generation Tables

In [ ]:
from haluguard.eval_matrix import run_generation_method

METRICS = [
    'em', 'em_normalized', 'es', 'codebleu', 'bleu_4', 'chrf_plus', 'rouge_l',
    'identifier_precision', 'identifier_recall', 'identifier_f1', 'identifier_em',
    'codebert_score', 'syntax_valid', 'non_empty_rate', 'avg_prediction_chars',
]

def run_table(benchmark_name, loader, plans):
    rows = []
    for plan in plans:
        print(f'[{benchmark_name}] {plan.method} ({plan.status})')
        summary = run_generation_method(
            plan=plan,
            loader=loader,
            generator=generator,
            output_dir=DETAILS,
            repobench_artifacts=repobench_artifacts,
            cceval_artifacts=cceval_artifacts,
            checkpoint_dirs=CHECKPOINT_DIRS,
            metrics=METRICS,
            limit=LIMIT,
            batch_size=BATCH_SIZE,
            top_k=TOP_K,
            device=DEVICE,
            encoder=metric_encoder,
            tokenizer=metric_tokenizer,
        )
        rows.append(summary.to_row())
    rows = sorted(rows, key=lambda r: (r.get('status') != 'completed', -(r.get('em') or 0.0)))
    write_table_files(rows, RESULTS / f'{benchmark_name}_generation_table')
    return rows

repobench_generation_rows = run_table('repobench', repobench_loader, repobench_plans)
cceval_generation_rows = run_table('crosscodeeval', cceval_loader, cceval_plans)

try:
    import pandas as pd
    display(pd.DataFrame(repobench_generation_rows))
    display(pd.DataFrame(cceval_generation_rows))
except ImportError:
    print(repobench_generation_rows)
    print(cceval_generation_rows)

## 8. Router Delta Table

In [ ]:
from haluguard.eval_matrix import build_router_delta_table

router_delta_rows = build_router_delta_table(repobench_generation_rows + cceval_generation_rows)
write_table_files(router_delta_rows, RESULTS / 'router_delta_table')
try:
    import pandas as pd
    display(pd.DataFrame(router_delta_rows))
except ImportError:
    print(router_delta_rows)

## 9. Example Analysis

In [ ]:
def read_details(method, benchmark='repobench'):
    path = DETAILS / f'{benchmark}__{method}.jsonl'
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def exact(pred, ref):
    return pred.strip() == ref.strip()

bm25 = read_details('bm25')
cosine = read_details('cosine_unixcoder_last3')
completed_halu = [r for r in repobench_generation_rows if r['status'] == 'completed' and '__noop' in r['method']]
if completed_halu:
    best = max(completed_halu, key=lambda r: r.get('em') or 0.0)['method']
    halu = read_details(best)
    wins = []
    for h, b in zip(halu, bm25):
        if exact(h['prediction'], h['reference']) and not exact(b['prediction'], b['reference']):
            wins.append({'source_index': h['source_index'], 'gt': h['reference'], 'haluguard': h['prediction'], 'bm25': b['prediction']})
    print('best HaluGuard noop:', best)
    print('HaluGuard-over-BM25 exact-match wins:', len(wins))
    for row in wins[:5]:
        print(row)
else:
    print('No completed HaluGuard noop rows available for example analysis.')

hurts = []
for delta in router_delta_rows:
    if delta.get('delta_em') is not None and delta['delta_em'] < 0:
        hurts.append(delta)
print('router rows with negative EM delta:', len(hurts))
for row in hurts[:10]:
    print(row)